<a href="https://colab.research.google.com/github/yulaAl20/eKYC-Report-Generator/blob/main/Copy_of_ekyc_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

upload just excel with just branch codes

In [ ]:
# Install dependencies
!pip install pandas openpyxl

import pandas as pd
from google.colab import files

# Upload Accounts file
print("Upload the Accounts Excel file:")
uploaded_accounts = files.upload()

# Upload Branches file
print("Upload the Branches Excel file:")
uploaded_branches = files.upload()

# Replace with actual uploaded file names
accounts_file = list(uploaded_accounts.keys())[0]
branches_file = list(uploaded_branches.keys())[0]

# Read Accounts and clean column names
df_accounts = pd.read_excel(accounts_file, dtype=str)
df_accounts.columns = df_accounts.columns.str.strip()  # remove spaces

# Read Branches and clean column names
df_branches = pd.read_excel(branches_file, dtype=str)
df_branches.columns = df_branches.columns.str.strip()  # remove spaces

# Count accounts by branch code
branch_counts = (
    df_accounts.groupby("Branch")
    .size()
    .reset_index(name="Account Count")
    .rename(columns={"Branch": "CODE"})
)

# Merge with branch names
final_df = branch_counts.merge(df_branches, on="CODE", how="left")

# Reorder columns
final_df = final_df[["CODE", "NAME", "Account Count"]]

# --- Find branches not in Accounts ---
missing_branches = df_branches[~df_branches["CODE"].isin(branch_counts["CODE"])]
missing_branches["Account Count"] = 0  # assign 0 for missing ones

# Combine all branches
combined_df = pd.concat([final_df, missing_branches[["CODE", "NAME", "Account Count"]]], ignore_index=True)

# --- Add total sum row at the end ---
total_accounts = combined_df["Account Count"].astype(int).sum()
total_row = pd.DataFrame({"CODE": ["TOTAL"], "NAME": [""], "Account Count": [total_accounts]})
combined_df = pd.concat([combined_df, total_row], ignore_index=True)

# Display result
print("\n✅ Branch-wise Account Summary:\n")
print(combined_df)

# Save to Excel
output_file = "branch_account_summary.xlsx"
combined_df.to_excel(output_file, index=False)

print(f"\n📁 Results saved to {output_file}")
files.download(output_file)



Upload the Accounts Excel file:


Saving OnboardingRequest-Detail_Report(15).xlsx to OnboardingRequest-Detail_Report(15).xlsx
Upload the Branches Excel file:


Saving brancheswithcodes.xlsx to brancheswithcodes (1).xlsx

✅ Branch-wise Account Summary:

      CODE                     NAME  Account Count
0     0012             Staff branch              4
1     0040               Maharagama              2
2     0050                 Panadura              1
3     0070                Ratnapura              1
4     0110                  Gampaha              2
..     ...                      ...            ...
171   1720                Hettipola              0
172   1730              Eheliyagoda              0
173   1760                  Padukka              0
174   0321  Priority Banking Centre              0
175  TOTAL                                     124

[176 rows x 3 columns]

📁 Results saved to branch_account_summary.xlsx


/tmp/ipykernel_690/3346660849.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_branches["Account Count"] = 0  # assign 0 for missing ones


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================
# MYSIS vs eKYC Report
# =========================

import pandas as pd
import re
from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

# -------- Upload files --------
print("Upload MYSIS file (Branch, Count)")
mysis_file = files.upload()
mysis_path = next(iter(mysis_file.keys()))

print("\nUpload eKYC file (CODE, NAME, Account Count)")
ekyc_file = files.upload()
ekyc_path = next(iter(ekyc_file.keys()))

# -------- Read as STRING (preserve leading zeros) --------
mysis = pd.read_excel(mysis_path, dtype=str)
ekyc  = pd.read_excel(ekyc_path, dtype=str)

mysis.columns = [str(c).strip() for c in mysis.columns]
ekyc.columns  = [str(c).strip() for c in ekyc.columns]

def find_col(df, candidates):
    cols = {c.upper(): c for c in df.columns}
    for cand in candidates:
        if cand.upper() in cols:
            return cols[cand.upper()]
    raise ValueError(f"Missing column. Need one of {candidates}")

m_branch_col = find_col(mysis, ["Branch", "Branch Code", "CODE"])
m_count_col  = find_col(mysis, ["Count"])

e_code_col   = find_col(ekyc, ["CODE", "Branch Code"])
e_name_col   = find_col(ekyc, ["NAME"])
e_count_col  = find_col(ekyc, ["Account Count", "Count"])

# -------- Normalize branch codes (KEEP leading zeros) --------
def normalize_code(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):   # fix Excel float artifact
        s = s[:-2]
    return s

mysis["_code"] = mysis[m_branch_col].apply(normalize_code)
ekyc["_code"]  = ekyc[e_code_col].apply(normalize_code)

# Remove any TOTAL rows from inputs
BAD = {"TOTAL", "GRAND TOTAL", "SUB TOTAL", "SUBTOTAL"}
mysis = mysis[~mysis["_code"].str.upper().isin(BAD)]
ekyc  = ekyc[~ekyc["_code"].str.upper().isin(BAD)]

# -------- Numeric counts --------
mysis["mysis_count"] = pd.to_numeric(mysis[m_count_col], errors="coerce").fillna(0).astype(int)
ekyc["ekyc_count"]   = pd.to_numeric(ekyc[e_count_col], errors="coerce").fillna(0).astype(int)

# -------- Aggregate --------
mysis_agg = mysis.groupby("_code", as_index=False)["mysis_count"].sum()

def first_non_empty(series):
    for v in series.dropna().astype(str):
        if v.strip():
            return v.strip()
    return ""

ekyc_agg = (
    ekyc.groupby("_code", as_index=False)
        .agg(
            Branch_name=(e_name_col, first_non_empty),
            ekyc_count=("ekyc_count", "sum")
        )
)

# -------- Merge --------
report = mysis_agg.merge(ekyc_agg, on="_code", how="outer")
report["mysis_count"] = report["mysis_count"].fillna(0).astype(int)
report["ekyc_count"]  = report["ekyc_count"].fillna(0).astype(int)
report["Branch_name"] = report["Branch_name"].fillna("")

report = report.rename(columns={
    "_code": "Branch code",
    "Branch_name": "Branch name"
})

report = report[["Branch code", "Branch name", "ekyc_count", "mysis_count"]]
report = report.sort_values("Branch name", na_position="last")

# -------- TOTAL row --------
total = pd.DataFrame([{
    "Branch code": "TOTAL",
    "Branch name": "",
    "ekyc_count":  report["ekyc_count"].sum(),
    "mysis_count": report["mysis_count"].sum()

}])

final_report = pd.concat([report, total], ignore_index=True)

# -------- Save Excel --------
out_file = "mysis_vs_ekyc_report.xlsx"
final_report.to_excel(out_file, index=False, sheet_name="Report")

# -------- Formatting --------
wb = load_workbook(out_file)
ws = wb["Report"]

header_fill = PatternFill("solid", fgColor="1F4E79")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")
    cell.border = border

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.border = border
        if cell.column in [3, 4]:
            cell.alignment = Alignment(horizontal="right")

# Highlight TOTAL row
for cell in ws[ws.max_row]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill("solid", fgColor="FFF2CC")

# Auto column width
for col in range(1, ws.max_column + 1):
    col_letter = get_column_letter(col)
    max_len = max(len(str(c.value)) if c.value else 0 for c in ws[col_letter])
    ws.column_dimensions[col_letter].width = min(max_len + 2, 40)

wb.save(out_file)

# -------- Download --------
files.download(out_file)

print("✅ Report generated successfully")


Upload MYSIS file (Branch, Count)


Saving Copy of 17032026_Summ.xlsx to Copy of 17032026_Summ.xlsx

Upload eKYC file (CODE, NAME, Account Count)


Saving 17-13-26.xlsx to 17-13-26.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Report generated successfully


In [ ]:
import pandas as pd
from google.colab import files

# Upload files
uploaded = files.upload()

# Get uploaded file names dynamically
file_names = list(uploaded.keys())

if len(file_names) != 2:
    raise Exception("Please upload exactly 2 Excel files.")

# Read both files
file1 = pd.read_excel(file_names[0])
file2 = pd.read_excel(file_names[1])

# Clean column names (removes spacing issues)
file1.columns = file1.columns.str.strip()
file2.columns = file2.columns.str.strip()

# Merge on CODE and NAME
merged = pd.merge(
    file1,
    file2,
    on=["CODE", "NAME"],
    how="inner",
    suffixes=("_FILE1", "_FILE2")
)

# Compare Account Count
differences = merged[
    merged["Account Count_FILE1"] != merged["Account Count_FILE2"]
]

print("Rows where Account Count is different:")
display(differences)

# Save result
differences.to_excel("account_count_differences.xlsx", index=False)

Saving branch_account_summary(6).xlsx to branch_account_summary(6) (1).xlsx
Saving 16.xlsx to 16.xlsx
Rows where Account Count is different:


,CODE,NAME,Account Count_FILE1,Account Count_FILE2
11,0300,Colombo Fort,18,0
65,1750,Union Place,5,6
175,TOTAL,NaN,130,113
